<a href="https://colab.research.google.com/github/Vengalagagan/NLP/blob/main/2403A52222(B_09)NLP_Lab_15_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import Libraries

In [62]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
import numpy as np

Load ELMO Model

In [63]:
elmo = hub.load("https://tfhub.dev/google/elmo/3")

Text Corpus(Documents)

In [64]:
sentences = [
    "The bank will not approve the loan",
    "He sat on the river bank",
    "She deposited money in the bank",
    "Children played near the bank of the lake",
    "The company borrowed funds from the bank",
    "The fisherman stood quietly on the bank",
    "The bank offers low interest rates",
    "Flowers were growing along the river bank"
]

Embeddings

In [65]:
embeddings = elmo.signatures['default'](tf.constant(sentences))['elmo']
print(embeddings.shape)

(8, 8, 1024)


Word Embedding (“bank”)

In [66]:
# Convert sentences into tokens
tokenized = [sentence.split() for sentence in sentences]

# Find index of "bank" in every sentence
bank_indices = [tokens.index("bank") for tokens in tokenized]

# Extract embeddings of "bank" from all sentences
bank_embeddings = [
    embeddings[i][bank_indices[i]]
    for i in range(len(sentences))
]

# Print first 10 values for each sentence
for i, emb in enumerate(bank_embeddings):
    print(f"Sentence {i+1}: {sentences[i]}")
    print("First 10 values:", emb[:10])
    print()

Sentence 1: The bank will not approve the loan
First 10 values: tf.Tensor(
[-0.9086766  -0.36578724 -0.0733975   0.5866749   0.22132955 -0.7657322
 -0.11816446 -0.3022788  -0.06137528  0.16951814], shape=(10,), dtype=float32)

Sentence 2: He sat on the river bank
First 10 values: tf.Tensor(
[-0.15614772  0.28345677 -0.08914644  0.20695065  0.15241832 -0.20373058
 -0.10812794  0.03747281  0.52148664  0.353526  ], shape=(10,), dtype=float32)

Sentence 3: She deposited money in the bank
First 10 values: tf.Tensor(
[-0.33989906  0.34286597 -0.07132763  0.36118096 -0.28549266 -0.31172186
 -0.41095564 -0.11300926  0.03143679  0.57589567], shape=(10,), dtype=float32)

Sentence 4: Children played near the bank of the lake
First 10 values: tf.Tensor(
[-0.63130385  0.36684275 -0.50689137  0.06840359 -0.14123407  0.04119962
 -0.29742515  0.08263844  0.69130445  0.03459672], shape=(10,), dtype=float32)

Sentence 5: The company borrowed funds from the bank
First 10 values: tf.Tensor(
[-0.49415144 -

Cosine Similarity

In [67]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Compare all bank embeddings with each other
for i in range(len(bank_embeddings)):
    for j in range(i + 1, len(bank_embeddings)):
        sim = cosine_similarity(
            bank_embeddings[i].numpy(),
            bank_embeddings[j].numpy()
        )
        print(f"Similarity between Sentence {i+1} and Sentence {j+1}: {sim:.4f}")

Similarity between Sentence 1 and Sentence 2: 0.5781
Similarity between Sentence 1 and Sentence 3: 0.6263
Similarity between Sentence 1 and Sentence 4: 0.6505
Similarity between Sentence 1 and Sentence 5: 0.6395
Similarity between Sentence 1 and Sentence 6: 0.5818
Similarity between Sentence 1 and Sentence 7: 0.9374
Similarity between Sentence 1 and Sentence 8: 0.5703
Similarity between Sentence 2 and Sentence 3: 0.7751
Similarity between Sentence 2 and Sentence 4: 0.6787
Similarity between Sentence 2 and Sentence 5: 0.7241
Similarity between Sentence 2 and Sentence 6: 0.8938
Similarity between Sentence 2 and Sentence 7: 0.5951
Similarity between Sentence 2 and Sentence 8: 0.9196
Similarity between Sentence 3 and Sentence 4: 0.6517
Similarity between Sentence 3 and Sentence 5: 0.8805
Similarity between Sentence 3 and Sentence 6: 0.7599
Similarity between Sentence 3 and Sentence 7: 0.6456
Similarity between Sentence 3 and Sentence 8: 0.7728
Similarity between Sentence 4 and Sentence 5: 

# BERT MODEL BANK

In [68]:
sent=[
    "The bank will not approve the loan",
    "He sat on the river bank"
]


In [69]:
preprocess= hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
bert_model = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3")

PREPROCESS INPUT

In [70]:
input=preprocess(sent)
print(input.keys())

dict_keys(['input_type_ids', 'input_mask', 'input_word_ids'])


GENERATE EMBEDDINGS

In [71]:
outputs=bert_model(input)
embeddings=outputs['sequence_output']
print("Embedding shape: ",embeddings.shape)

Embedding shape:  (2, 128, 768)


EXREACT BANK EMBEDDINGS

In [72]:
bank_emb_1 = embeddings[0][2]
bank_emb_2 = embeddings[1][6]
print("First 10 values (Sentence 1):", bank_emb_1[:10])
print("First 10 values (Sentence 2):", bank_emb_2[:10])

First 10 values (Sentence 1): tf.Tensor(
[ 0.29043403 -0.32690707  0.4144945   0.29487795  1.2237236   0.18653788
 -0.49868098  0.6392791   0.09137864  0.15223756], shape=(10,), dtype=float32)
First 10 values (Sentence 2): tf.Tensor(
[-0.5490642  -0.67239016 -0.70398504  0.00643293 -0.57983094  0.31554186
  0.22425303  0.8700546  -0.11308757 -0.91464883], shape=(10,), dtype=float32)


Get Tokens

In [73]:
tokenizer = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
tokens=preprocess(sent)['input_word_ids']
print(tokens)

tf.Tensor(
[[  101  1996  2924  2097  2025 14300  1996  5414   102     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0]
 [  101  2002  2938  2006  1996  2314  2924   102     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0 

COSINE SIMILARITY

In [100]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim = cosine_similarity(bank_emb_1.numpy(), bank_emb_2.numpy())

print("Similarity between 'bank' meanings:", sim)

Similarity between 'bank' meanings: 0.30439484


# **Task - II**

# ELMO BAT

In [75]:
text=[
    "The bat iss flying",
    "He hit the ball with a bat"
]

GENERATE EMBEDDINGS

In [76]:
embedding = elmo.signatures['default'](tf.constant(text))['elmo']
print(embedding.shape)

(2, 7, 1024)


 INSPECT WORD EMBEDDING ("BAT")

In [77]:
tokenized = [sentences.split() for sentences in text]
idx1=tokenized[0].index("bat")
idx2=tokenized[1].index("bat")
bat_emb_1=embedding[0][idx1]
bat_emb_2=embedding[1][idx2]
print("First 10 values (sentence 1):",bat_emb_1[:10])
print("First 10 values (sentence 2):",bat_emb_2[:10])

First 10 values (sentence 1): tf.Tensor(
[-0.45903727 -0.27516434 -0.26348162 -0.12380227 -0.11530864 -0.45880768
 -0.12675564 -0.12455821 -0.2630463  -0.1839562 ], shape=(10,), dtype=float32)
First 10 values (sentence 2): tf.Tensor(
[-0.20741455  0.01774421  0.12503807 -0.28234187 -0.23928387  0.04694621
 -0.08645728  0.31607887 -0.13618736 -0.05437148], shape=(10,), dtype=float32)


cosine similarity

In [78]:
def cosine_similarity(A,B):
  return np.dot(A,B)/(np.linalg.norm(A)*np.linalg.norm(B))
simi = cosine_similarity(bat_emb_1.numpy(),bat_emb_2.numpy())
print("Similarity between 'bat' meanings:",simi)

Similarity between 'bat' meanings: 0.6627627


##BERT Model bat

In [79]:
sentences = [
    "The bat is flying",
    "He hit the ball with a bat"
]

| Component           | Role                          |
| ------------------- | ----------------------------- |
| Preprocessing model | Converts text → numbers       |
| BERT encoder        | Converts numbers → embeddings |


**Preprocessing Model**
Lowercasing
Tokenization
Add Special Tokens
Convert words → IDs
Create inputs required by BERT

(batch_size, sequence_length, 768)

In [80]:
# Preprocessing model
preprocess = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")

# BERT encoder
bert_model = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3")

Input

In [81]:
inputs = preprocess(sentences)
print(inputs.keys())

dict_keys(['input_type_ids', 'input_mask', 'input_word_ids'])


Generate Embeddings

In [82]:
outputs = bert_model(inputs)

# Word-level embeddings
embeddings = outputs['sequence_output']

print("Embedding shape:", embeddings.shape)

Embedding shape: (2, 128, 768)


Get Tokens

In [83]:
# Use tokenizer from preprocess model
tokenizer = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")

# Tokenized words (for visualization)
tokens = preprocess(sentences)['input_word_ids']

print(tokens)

tf.Tensor(
[[ 101 1996 7151 2003 3909  102    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0]
 [ 101 2002 2718 1996 3608 2007 1037 7151  102    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    

 bat Embeddings

In [84]:
bat_emb_1 = embeddings[0][2]
bat_emb_2 = embeddings[1][7]
print("First 10 values (Sentence 1):", bat_emb_1[:10])
print("First 10 values (Sentence 2):", bat_emb_2[:10])

First 10 values (Sentence 1): tf.Tensor(
[-6.0753897e-05  2.6044556e-01  1.2677932e-01 -2.8091615e-01
  1.3292141e-02 -4.8430157e-01  7.0899057e-01  7.3061734e-01
  2.0883092e-01 -7.8647351e-01], shape=(10,), dtype=float32)
First 10 values (Sentence 2): tf.Tensor(
[-0.06056846 -0.9072489  -0.06744834 -0.2745369  -0.46633822  0.04165892
  0.23526241  0.3572899   0.91712844 -0.67234397], shape=(10,), dtype=float32)


Cosine Similarity

In [85]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim = cosine_similarity(bat_emb_1.numpy(), bat_emb_2.numpy())

print("Similarity between 'bat' meanings:", sim)

Similarity between 'bat' meanings: 0.43427742


##Text Classification Using ELMO+Naive Bayes

Import Libraries

In [86]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np

from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# **Text Corpus**

In [87]:
sentences = test_samples = [
    "Congratulations you won a gift card",
    "Urgent! Verify your bank account now",
    "Did you finish the assignment",
    "We will go to college tomorrow",
    "Limited time offer click here",
    "Can you send me the notes"
]

labels = [1, 1, 0, 0, 1, 0]

Load ELMo Model

In [88]:
elmo = hub.load("https://tfhub.dev/google/elmo/3")

Generate ELMo Embeddings

In [89]:
embeddings = elmo.signatures['default'](tf.constant(sentences))['elmo']

print("Shape:", embeddings.shape)

Shape: (6, 6, 1024)


 Sentence Embeddings

In [90]:
sentence_embeddings = tf.reduce_mean(embeddings, axis=1)

X = sentence_embeddings.numpy()
y = np.array(labels)

print("Feature shape:", X.shape)

Feature shape: (6, 1024)


Train-Test & Split

In [91]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Train Model

In [92]:
model = GaussianNB()
model.fit(X_train, y_train)

GaussianNB()

Model Testing

In [93]:
y_pred = model.predict(X_test)

print("Predictions:", y_pred)
print("Actual:", y_test)

Predictions: [0 0]
Actual: [1 1]


Model Evaluation

In [94]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.0


Prediction on New Text

In [95]:
new_sentence = ["Congratulations! You won a free ticket"]

new_emb = elmo.signatures['default'](tf.constant(new_sentence))['elmo']
new_emb = tf.reduce_mean(new_emb, axis=1)

prediction = model.predict(new_emb.numpy())

print("Prediction:", "Spam" if prediction[0] == 1 else "Not Spam")

Prediction: Not Spam


##Text Classification using BERT+NB

In [96]:
preprocess = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
bert_model = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3")

In [97]:
bert_inputs = preprocess(sentences)

In [98]:
bert_outputs = bert_model(bert_inputs)

# Use sentence embedding ([CLS])
bert_features = bert_outputs['pooled_output'].numpy()

In [99]:
X_train, X_test, y_train, y_test = train_test_split(
    bert_features, labels, test_size=0.2, random_state=42
)

nb_bert = GaussianNB()
nb_bert.fit(X_train, y_train)

y_pred_bert = nb_bert.predict(X_test)
acc_bert = accuracy_score(y_test, y_pred_bert)

print("BERT Accuracy:", acc_bert)

BERT Accuracy: 0.0
